<a href="https://colab.research.google.com/github/ysugiyama3/google_colab/blob/master/cataloging_copy_lookup_wcapi_2_yale_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **🍕Cataloging copy lookup (FOR YALE STAFF ONLY - DO NOT SHARE)🌮**

The program automatically searches for good OCLC records based on LCCN, ISBN, and OCLC numbers, using the WorldCat Search API ver. 2.

**What you need**

* Excel spreadsheet in which **the 1st, 2nd, 3rd columns must be assigned for LCCN, ISBN, and OCLC number respectively**. The spreadsheet can have as many columns as necesary and must have column headers.

**How to run the program**
* Simply click the play button in the order.
* To start over, please go to the menu, go to "Runtime" and then select "Disconnect and delete runtime."
* To clear output, please go to "Edit" and then select "Clear all outputs."

Contact yukari.sugiyama@yale.edu if you have any issues or questions. (Last updated 10/9/2025)

---

In [ ]:
#@title <--- Upload Excel file and Search OCLC WorldCat

from google.colab import files, userdata
import pandas as pd
import requests
import json
from IPython.display import HTML, display
import time
import re
import matplotlib.pyplot as plt
from typing import Optional, Dict, List, Tuple
import logging
import base64

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuration Constants
COLUMN_NAMES = {
    'LCCN': 0,  # Column index for LCCN
    'ISBN': 1,  # Column index for ISBN
    'OCLC': 2   # Column index for OCLC
}

OUTPUT_COLUMNS = ['[MATCH_OCLC_NO]', '[SUBJECT]', '[LC_CALL_NUMBER]', '[ELVL]']

API_CONFIG = {
    'base_url': 'https://americas.discovery.api.oclc.org/worldcat/search/v2',
    'token_url': 'https://oauth.oclc.org/token?grant_type=client_credentials&scope=wcapi',
    'rate_limit_delay': 0.02,  # seconds between requests
    'token_refresh_interval': 900,  # refresh token every 15 minutes
    'search_params': 'inCatalogLanguage=eng&itemSubType=book-printbook&orderBy=bestMatch&limit=1'
}


class ProgressBar:
    """Handle progress bar display in Colab"""

    def __init__(self, total: int):
        self.total = total
        self.display_handle = None

    def show(self, current: int):
        """Update progress bar"""
        html = f"""
            <progress value='{current}' max='{self.total}' style='width: 40%'>
                {current}
            </progress>
            <br>{current}/{self.total}</br>
        """
        if self.display_handle is None:
            self.display_handle = display(HTML(html), display_id=True)
        else:
            self.display_handle.update(HTML(html))


class DataCleaner:
    """Clean and normalize bibliographic identifiers"""

    @staticmethod
    def clean_lccn(lccn) -> Optional[str]:
        """Clean LCCN number"""
        if pd.isnull(lccn) or str(lccn).strip() == '':
            return None

        lccn_str = str(lccn).strip()
        if len(lccn_str) > 0:
            # Remove everything after the period
            return re.sub(r'\..*', '', lccn_str).strip()
        return None

    @staticmethod
    def clean_isbn(isbn) -> Optional[str]:
        """Clean ISBN number"""
        if pd.isnull(isbn) or str(isbn).strip() == '':
            return None

        isbn_str = str(isbn)
        # Remove parentheses, colons, periods and everything after
        isbn_str = re.sub(r'[\(|\:|\.].*', '', isbn_str)
        # Keep only digits and X
        isbn_str = re.sub(r'[^0-9Xx]', '', isbn_str).strip()

        return isbn_str if isbn_str else None

    @staticmethod
    def clean_oclc(oclc) -> Optional[str]:
        """Clean OCLC number"""
        if pd.isnull(oclc) or str(oclc).strip() == '':
            return None

        oclc_str = str(oclc).strip()
        if len(oclc_str) == 0:
            return None

        # Check for OCLC prefixes
        if oclc_str.startswith(('(OCoLC)', 'ocn', 'ocm', 'on')):
            oclc_str = oclc_str.split(';')[0].strip()
            # Extract only digits
            oclc_str = re.sub(r'[^0-9]', '', oclc_str).strip()
            return oclc_str if oclc_str else None

        return None


class OCLCSearcher:
    """Handle OCLC WorldCat API searches"""

    def __init__(self, client_id: str, client_secret: str):
        self.client_id = client_id
        self.client_secret = client_secret
        self.token = None
        self.token_timestamp = 0
        self.stats = {'matched': 0, 'total': 0}
        self._refresh_token()

    def _refresh_token(self):
        """Get or refresh OAuth token"""
        try:
            # Create Basic auth header from client_id:client_secret
            credentials = f"{self.client_id}:{self.client_secret}"
            encoded_credentials = base64.b64encode(credentials.encode()).decode()
            headers = {'Authorization': f'Basic {encoded_credentials}'}

            response = requests.post(API_CONFIG['token_url'], headers=headers)
            response.raise_for_status()

            data = response.json()
            self.token = data.get('access_token')
            self.token_timestamp = time.time()
            logger.info("Token refreshed successfully")
        except requests.RequestException as e:
            logger.error(f"Failed to get token: {e}")
            raise

    def _should_refresh_token(self) -> bool:
        """Check if token needs refresh"""
        return (time.time() - self.token_timestamp) > API_CONFIG['token_refresh_interval']

    def _make_request(self, url: str) -> Optional[Dict]:
        """Make API request with error handling"""
        if self._should_refresh_token():
            self._refresh_token()

        headers = {'Authorization': f'Bearer {self.token}'}

        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            return response.json()
        except requests.Timeout:
            logger.warning(f"Request timeout for URL: {url}")
            return None
        except requests.RequestException as e:
            logger.warning(f"Request failed: {e}")
            return None

    def _extract_bib_data(self, data: Dict) -> Dict[str, str]:
        """Extract bibliographic information from API response"""
        result = {
            'oclc': '',
            'lcsh': 'No',
            'call_no': '',
            'elvl': ''
        }

        try:
            bib_record = data['bibRecords'][0]

            # OCLC number
            result['oclc'] = bib_record.get('identifier', {}).get('oclcNumber', '')

            # Subjects and material type
            subjects = bib_record.get('subjects', [])
            material_types = bib_record.get('format', {}).get('materialTypes', [])
            result['lcsh'] = self._check_lcsh(subjects, material_types)

            # Call number
            result['call_no'] = bib_record.get('classification', {}).get('lc', '')

            # Get encoding level if we have OCLC number
            if result['oclc']:
                result['elvl'] = self._get_encoding_level(result['oclc'])

        except (KeyError, IndexError, TypeError) as e:
            logger.debug(f"Error extracting bib data: {e}")

        return result

    def _check_lcsh(self, subjects: List[Dict], material_types: List[str]) -> str:
        """Check if record has LCSH"""
        for subject in subjects:
            if subject.get('vocabulary') == 'Library of Congress Subject Headings':
                return 'Yes'

        # Check if it's fiction
        if any('fic' in mt.lower() for mt in material_types):
            return 'Yes (Fiction)'

        return 'No'

    def _get_encoding_level(self, oclc: str) -> str:
        """Get encoding level for OCLC number"""
        url = f"{API_CONFIG['base_url']}/brief-bibs/{oclc}"
        data = self._make_request(url)

        if data:
            try:
                return data.get('catalogingInfo', {}).get('levelOfCataloging', '?')
            except (KeyError, TypeError):
                pass

        return '?'

    def search_by_identifier(self, identifier: Optional[str], search_type: str) -> Optional[Dict[str, str]]:
        """
        Search OCLC by identifier

        Args:
            identifier: The identifier value (LCCN, ISBN, or OCLC)
            search_type: Type of search ('dn' for LCCN, 'bn' for ISBN, 'no' for OCLC)

        Returns:
            Dictionary with bibliographic data or None if not found
        """
        if not identifier:
            return None

        url = f"{API_CONFIG['base_url']}/bibs?q={search_type}:{identifier}&{API_CONFIG['search_params']}"
        data = self._make_request(url)

        if data and data.get('bibRecords'):
            return self._extract_bib_data(data)

        return None

    def search_cascade(self, lccn: Optional[str], isbn: Optional[str],
                       oclc: Optional[str]) -> Dict[str, str]:
        """
        Try searching by LCCN, then ISBN, then OCLC

        Returns:
            Dictionary with bibliographic data (empty strings if nothing found)
        """
        # Try LCCN first
        result = self.search_by_identifier(lccn, 'dn')
        if result:
            self.stats['matched'] += 1
            return result

        # Try ISBN
        result = self.search_by_identifier(isbn, 'bn')
        if result:
            self.stats['matched'] += 1
            return result

        # Try OCLC
        result = self.search_by_identifier(oclc, 'no')
        if result:
            self.stats['matched'] += 1
            return result

        # Nothing found
        return {'oclc': '', 'lcsh': '', 'call_no': '', 'elvl': ''}


class ExcelProcessor:
    """Handle Excel file processing"""

    def __init__(self, searcher: OCLCSearcher, cleaner: DataCleaner):
        self.searcher = searcher
        self.cleaner = cleaner

    def process_file(self, input_df: pd.DataFrame) -> Tuple[pd.DataFrame, int]:
        """
        Process input DataFrame and add OCLC search results

        Returns:
            Tuple of (output_df, matched_count)
        """
        # Create output DataFrame with new columns
        output_df = input_df.copy()
        for col in OUTPUT_COLUMNS:
            output_df[col] = ''

        total = len(output_df)
        progress = ProgressBar(total)

        logger.info(f"Processing {total} records...")

        for index, row in output_df.iterrows():
            # Update progress
            progress.show(index + 1)

            # Clean identifiers
            lccn = self.cleaner.clean_lccn(row.iloc[COLUMN_NAMES['LCCN']])
            isbn = self.cleaner.clean_isbn(row.iloc[COLUMN_NAMES['ISBN']])
            oclc = self.cleaner.clean_oclc(row.iloc[COLUMN_NAMES['OCLC']])

            # Search OCLC
            result = self.searcher.search_cascade(lccn, isbn, oclc)

            # Update DataFrame
            output_df.loc[index, '[MATCH_OCLC_NO]'] = result['oclc']
            output_df.loc[index, '[SUBJECT]'] = result['lcsh']
            output_df.loc[index, '[LC_CALL_NUMBER]'] = result['call_no']
            output_df.loc[index, '[ELVL]'] = result['elvl']

            # Rate limiting
            time.sleep(API_CONFIG['rate_limit_delay'])

        return output_df, self.searcher.stats['matched']


def visualize_results(input_name: str, matched: int, total: int):
    """Create pie chart of results"""
    labels = 'Has copy', 'No copy'
    sizes = [matched, total - matched]
    colors = ['c', 'y']
    explode = (0.1, 0.0)

    plt.figure(figsize=(8, 6))
    plt.pie(sizes, explode=explode, labels=labels, colors=colors,
            autopct=lambda s: '{:.0f}'.format(s * total / 100),
            shadow=True, startangle=90)

    plt.axis('equal')
    plt.title(f'{input_name} Results\n{matched} matched out of {total} total')
    plt.show()


def save_output(output_df: pd.DataFrame, output_name: str) -> str:
    """Save output file (Excel or CSV fallback)"""
    try:
        output_df.to_excel(output_name, index=False)
        logger.info(f"Saved Excel file: {output_name}")
        return output_name
    except Exception as e:
        logger.warning(f"Excel save failed: {e}. Saving as CSV instead.")
        csv_name = output_name.rsplit(".", 1)[0] + '.csv'
        output_df.to_csv(csv_name, index=False, encoding='utf-8')
        logger.info(f"Saved CSV file: {csv_name}")
        return csv_name


def main():
    """Main execution function"""
    print("=" * 60)
    print("OCLC WorldCat Search Tool")
    print("=" * 60)

    # Get API credentials from Colab secrets
    try:
        client_id = userdata.get('OCLC_CLIENT_ID')
        client_secret = userdata.get('OCLC_CLIENT_SECRET')
        print("\n✓ API credentials loaded from secrets")
    except userdata.SecretNotFoundError as e:
        print(f"\n❌ Error: Required secret not found: {e}")
        print("\nTo add the secrets:")
        print("1. Click the 🔑 key icon in the left sidebar")
        print("2. Add these two secrets:")
        print("   - Name: OCLC_CLIENT_ID")
        print("     Value: Your OCLC client ID (e.g., rtJKed7UrR2d7zVW...)")
        print("   - Name: OCLC_CLIENT_SECRET")
        print("     Value: Your OCLC client secret (e.g., rlT4nuOF+6Qm...)")
        print("3. Toggle 'Notebook access' ON for this notebook")
        return
    except Exception as e:
        print(f"\n❌ Error accessing secrets: {e}")
        return

    # Upload file
    print("\n📤 Upload your Excel file...")
    uploaded = files.upload()

    if not uploaded:
        print("❌ Error: No file uploaded")
        return

    input_name = list(uploaded.keys())[0]
    print(f"✓ File uploaded: {input_name}")

    # Read input file
    try:
        input_df = pd.read_excel(input_name)
        print(f"✓ Read {len(input_df)} records from file")
    except Exception as e:
        print(f"❌ Error reading file: {e}")
        return

    # Validate input
    if len(input_df.columns) < 3:
        print("❌ Error: Excel file must have at least 3 columns (LCCN, ISBN, OCLC)")
        return

    # Initialize components
    try:
        searcher = OCLCSearcher(client_id, client_secret)
        cleaner = DataCleaner()
        processor = ExcelProcessor(searcher, cleaner)
    except Exception as e:
        print(f"❌ Error initializing: {e}")
        return

    # Process file
    print("\n🔍 Searching OCLC WorldCat...")
    try:
        output_df, matched = processor.process_file(input_df)
        total = len(input_df)

        print(f"\n✓ Processing complete!")
        print(f"  - Total records: {total}")
        print(f"  - Matched: {matched}")
        print(f"  - Not matched: {total - matched}")
        print(f"  - Match rate: {matched/total*100:.1f}%")

    except Exception as e:
        print(f"❌ Error during processing: {e}")
        logger.exception("Processing error")
        return

    # Save output
    output_name = input_name.rsplit(".", 1)[0] + "_output.xlsx"
    final_name = save_output(output_df, output_name)

    # Visualize
    print("\n📊 Generating visualization...")
    visualize_results(input_name, matched, total)

    # Download
    print("\n📥 Downloading output file...")
    files.download(final_name)

    print('\n✅ Done! 😎\n')


# Run the script
if __name__ == "__main__":
    main()